In [1]:
!pip install transformers datasets torch torchvision


  Using cached numpy-2.2.6-cp310-cp310-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached urllib

In [2]:
from transformers import AutoImageProcessor, SwinForImageClassification
from datasets import load_dataset
import torch

/Users/dhaneshkumarkapadia/Desktop/swinTransformer/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_name = "microsoft/swin-tiny-patch4-window7-224"
image_processor = AutoImageProcessor.from_pretrained(model_name)
model = SwinForImageClassification.from_pretrained(model_name)

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 233/233 [00:00<00:00, 1492.27it/s, Materializing param=swin.layernorm.weight]                                                     


In [4]:
dataset = load_dataset("cifar10", split="test[:8]")

In [5]:
images = [item["img"] for item in dataset]
labels = [item["label"] for item in dataset]

In [6]:
inputs = image_processor(images, return_tensors="pt").to(model.device)

In [10]:
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

In [11]:
predicted_labels = logits.argmax(dim=-1).cpu().numpy()

In [12]:
num_classes = len(model.config.id2label)
if num_classes != len(set(labels)):
    print("Warning: Model label space does not match CIFAR-10 labels. Mapping may be required.")
    class_mapping = {i: i % 10 for i in range(num_classes)}
    predicted_labels = [class_mapping[label] for label in predicted_labels]

In [13]:
class_names = [
    "airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"
]
predicted_class_names = [class_names[label] for label in predicted_labels]
true_class_names = [class_names[label] for label in labels]

print(predicted_class_names)
print(true_class_names)

['dog', 'automobile', 'ship', 'bird', 'bird', 'ship', 'dog', 'automobile']
['cat', 'ship', 'ship', 'airplane', 'frog', 'frog', 'automobile', 'frog']


In [14]:
for i, (true_label, predicted_label) in enumerate(zip(true_class_names, predicted_class_names)):
    print(
        f"Image {i + 1}: True Label = {true_label}, Predicted Label = {predicted_label}")

Image 1: True Label = cat, Predicted Label = dog
Image 2: True Label = ship, Predicted Label = automobile
Image 3: True Label = ship, Predicted Label = ship
Image 4: True Label = airplane, Predicted Label = bird
Image 5: True Label = frog, Predicted Label = bird
Image 6: True Label = frog, Predicted Label = ship
Image 7: True Label = automobile, Predicted Label = dog
Image 8: True Label = frog, Predicted Label = automobile


In [15]:
from sklearn.metrics import confusion_matrix, classification_report


cm = confusion_matrix(true_class_names, predicted_class_names)
print(cm)

print(classification_report(true_class_names, predicted_class_names))

[[0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0]
 [0 1 1 0 0 0 1]
 [0 1 0 0 0 0 1]]
              precision    recall  f1-score   support

    airplane       0.00      0.00      0.00         1
  automobile       0.00      0.00      0.00         1
        bird       0.00      0.00      0.00         0
         cat       0.00      0.00      0.00         1
         dog       0.00      0.00      0.00         0
        frog       0.00      0.00      0.00         3
        ship       0.50      0.50      0.50         2

    accuracy                           0.12         8
   macro avg       0.07      0.07      0.07         8
weighted avg       0.12      0.12      0.12         8



/Users/dhaneshkumarkapadia/Desktop/swinTransformer/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/dhaneshkumarkapadia/Desktop/swinTransformer/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/dhaneshkumarkapadia/Desktop/swinTransformer/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control th